In [18]:
# ------------------------------------------------------------
# Cell 1 — Setup / Imports / Paths
# ------------------------------------------------------------

from pathlib import Path
import sys
import importlib

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd()

# Make sure local modules in this folder are importable
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import flow_data_utils as fdu
import rating_curve_utils as rcu

# Reload while developing the utility module
importlib.reload(fdu)
importlib.reload(rcu)

print("flow_data_utils:", fdu.__file__)
print("rating_curve_utils:", rcu.__file__)

CACHE_PATH = ROOT / "cache" / "tributary_cache.joblib"
PLOTS_DIR = ROOT / "Plots_Discharge"
DATA_DIR = ROOT / "Data_Filled"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

SID = "07381000"
SHORT_GAP_DAYS = 3
BAYOU_LAFOURCHE_CHANGE_DATE = "2016-01-01"


In [25]:
# ------------------------------------------------------------
# Cell 2 — Load cached processed data
# ------------------------------------------------------------

if not CACHE_PATH.exists():
    raise FileNotFoundError(f"Cache not found: {CACHE_PATH}")

objs = joblib.load(CACHE_PATH)

station_metadata = objs.get("station_metadata", {}) or {}
station_names = objs.get("station_names", {}) or {}
flow_data_reindexed = objs.get("flow_data_reindexed", {}) or {}
filled_flow_data = objs.get("filled_flow_data", {}) or {}


In [36]:
# ------------------------------------------------------------
# Cell 3 — Shared helpers and refit-specific functions
# ------------------------------------------------------------

def _clean_name(s: str) -> str:
    return str(s).replace(" ", "_").replace(",", "").replace("/", "_")

def _safe_eval_rc_stage_only(rc_str: str, x_val: float) -> float:
    """For legacy stage-only formulas/plots."""
    if rc_str is None or str(rc_str).strip() == "" or pd.isna(x_val):
        return np.nan
    try:
        local = {"np": np, "H": float(x_val), "h": float(x_val), "Q": float(x_val)}
        return float(eval(str(rc_str), {"__builtins__": {}}, local))
    except Exception:
        return np.nan

# ---------- model fitting ----------
def _fit_linear(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        raise ValueError("Not enough finite points to fit line.")

    a, b = np.polyfit(np.asarray(x)[mask], np.asarray(y)[mask], 1)
    return float(a), float(b)

def _line_eval(a, b, x):
    return a * np.asarray(x) + b

def fit_hq_only_rc(Q_tgt: pd.Series, H: pd.Series, Qd: pd.Series):
    """Fit Q = a*H + b*Q + c*Q^2 + d using overlap days."""
    df = pd.concat({"H": H, "Qd": Qd, "Y": Q_tgt}, axis=1).dropna()
    n = len(df)
    if n < 12:
        raise ValueError(f"Too few overlap pairs for HQ fit (N={n}). Need >= 12.")

    Hn = df["H"].to_numpy()
    Qn = df["Qd"].to_numpy()
    Yn = df["Y"].to_numpy()

    X = np.column_stack([Hn, Qn, Qn**2, np.ones(n)])
    coefs, *_ = np.linalg.lstsq(X, Yn, rcond=None)

    a, b, c, d = [float(v) for v in coefs]
    rc = f"{a:.6g}*H + {b:.6g}*Q + {c:.6g}*Q**2 + {d:.6g}"

    yhat = a * Hn + b * Qn + c * (Qn**2) + d
    ss_res = float(np.sum((Yn - yhat)**2))
    ss_tot = float(np.sum((Yn - Yn.mean())**2))
    nse = np.nan if ss_tot == 0 else 1.0 - ss_res/ss_tot
    return rc, n, nse

# ---------- data loading ----------
def _load_base_series_only_original(objs, SID):
    """
    Return:
      Q_orig: original-only discharge series
      H_daily_from_csv: stage series from same station dataframe
      idx: Date index
    """
    filled_flow_data = objs.get("filled_flow_data", {}) or {}
    df = filled_flow_data.get(SID)
    if df is None or df.empty:
        raise ValueError(f"{SID}: missing filled flow data.")

    df = df.copy()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.set_index("Date")
    else:
        df.index = pd.to_datetime(df.index, errors="coerce")
    df = df.sort_index()
    idx = df.index

    if "Discharge" not in df.columns:
        raise ValueError(f"{SID}: missing Discharge column.")

    if "Discharge_filled_method" in df.columns:
        method = df["Discharge_filled_method"].astype(str).str.strip().str.lower()
        original_mask = (method.eq("") | method.eq("original")) & df["Discharge"].notna()
    else:
        original_mask = df["Discharge"].notna()

    Q_orig = pd.to_numeric(df["Discharge"], errors="coerce").where(original_mask)

    if "Stage" in df.columns:
        H_daily_from_csv = pd.to_numeric(df["Stage"], errors="coerce")
    else:
        H_daily_from_csv = pd.Series(index=idx, dtype=float)

    return Q_orig, H_daily_from_csv, idx

def _get_donor_series(objs, SID, idx, H_daily_from_csv):
    """
    Return:
      H: donor stage predictor (reindexed to idx)
      Qd: donor discharge predictor (reindexed to idx)
      donor_sid: donor station id
      donor_df: donor dataframe indexed by Date
    """
    station_metadata = objs.get("station_metadata", {}) or {}
    flow_data_reindexed = objs.get("flow_data_reindexed", {}) or {}

    meta = station_metadata.get(SID, {}) or {}
    donor_sid = (meta.get("Station ID used for rating curve") or "").strip()
    if not donor_sid:
        raise ValueError(f"{SID}: missing donor SID in metadata.")

    donor = flow_data_reindexed.get(donor_sid)
    if donor is None or donor.empty:
        raise ValueError(f"{SID}: donor station '{donor_sid}' not found.")

    donor = donor.copy()
    if "Date" in donor.columns:
        donor["Date"] = pd.to_datetime(donor["Date"], errors="coerce")
        donor = donor.set_index("Date")
    else:
        donor.index = pd.to_datetime(donor.index, errors="coerce")
    donor = donor.sort_index()

    if "Stage" in donor.columns:
        H = pd.to_numeric(donor["Stage"], errors="coerce").reindex(idx)
    else:
        H = pd.to_numeric(H_daily_from_csv, errors="coerce").reindex(idx)

    if "Discharge" in donor.columns:
        Qd = pd.to_numeric(donor["Discharge"], errors="coerce").reindex(idx)
    else:
        Qd = pd.Series(index=idx, dtype=float)

    return H, Qd, donor_sid, donor

def _build_daily_original_target_and_donor_hq(objs, SID, start, end, prefer_dv=True, short_gap_days=3):
    """
    Build daily original target discharge and donor H/Q predictors.
    """
    flow_data_reindexed = objs.get("flow_data_reindexed", {}) or {}
    filled_flow_data = objs.get("filled_flow_data", {}) or {}
    station_metadata = objs.get("station_metadata", {}) or {}

    if SID not in filled_flow_data:
        raise ValueError(f"{SID}: target not found in filled_flow_data.")

    meta = station_metadata.get(SID, {}) or {}
    donor_sid = (meta.get("Station ID used for rating curve") or "").strip() or SID

    if donor_sid not in flow_data_reindexed:
        raise ValueError(f"{SID}: donor '{donor_sid}' not found in flow_data_reindexed.")

    # target originals only
    df_t = filled_flow_data[SID].copy()
    if "Date" in df_t.columns:
        df_t["Date"] = pd.to_datetime(df_t["Date"], errors="coerce")
        df_t = df_t.set_index("Date").sort_index()
    else:
        df_t.index = pd.to_datetime(df_t.index, errors="coerce")
        df_t = df_t.sort_index()

    if "Discharge" not in df_t.columns:
        raise ValueError(f"{SID}: target Discharge missing.")

    method = df_t.get("Discharge_filled_method", pd.Series(index=df_t.index, dtype=object)).astype(str).str.strip().str.lower()
    orig = (method.eq("") | method.eq("original")) & df_t["Discharge"].notna()

    if prefer_dv and "Discharge_source" in df_t.columns:
        sel = orig & df_t["Discharge_source"].eq("daily")
        if not sel.any():
            sel = orig
    else:
        sel = orig

    q = pd.to_numeric(df_t.loc[sel, "Discharge"], errors="coerce")
    q = q.loc[(q.index >= pd.Timestamp(start)) & (q.index <= pd.Timestamp(end))]
    Q_orig = q.groupby(q.index.normalize()).mean().reindex(pd.date_range(start, end, freq="D"))

    # donor
    df_d = flow_data_reindexed[donor_sid].copy()
    if "Date" in df_d.columns:
        df_d["Date"] = pd.to_datetime(df_d["Date"], errors="coerce")
        df_d = df_d.set_index("Date").sort_index()
    else:
        df_d.index = pd.to_datetime(df_d.index, errors="coerce")
        df_d = df_d.sort_index()

    H = pd.to_numeric(df_d.get("Stage"), errors="coerce")
    H = H.loc[(H.index >= pd.Timestamp(start)) & (H.index <= pd.Timestamp(end))]
    H = H.groupby(H.index.normalize()).mean().reindex(pd.date_range(start, end, freq="D"))
    H = H.interpolate(limit=3, limit_direction="both", limit_area="inside")

    if "Discharge" in df_d.columns:
        if prefer_dv and "Discharge_source" in df_d.columns:
            qd = pd.to_numeric(df_d.loc[df_d["Discharge_source"].eq("daily"), "Discharge"], errors="coerce")
            if qd.dropna().empty:
                qd = pd.to_numeric(df_d["Discharge"], errors="coerce")
        else:
            qd = pd.to_numeric(df_d["Discharge"], errors="coerce")
    else:
        qd = pd.Series(index=df_d.index, dtype=float)

    qd = qd.loc[(qd.index >= pd.Timestamp(start)) & (qd.index <= pd.Timestamp(end))]
    Qd = qd.groupby(qd.index.normalize()).mean().reindex(pd.date_range(start, end, freq="D"))
    Qd = Qd.interpolate(limit=3, limit_direction="both", limit_area="inside")

    return Q_orig, H, Qd, donor_sid

# ---------- refit-specific fill engine ----------
def fill_all_missing_with_split_models(Q_orig: pd.Series, X: pd.Series, change_ts: pd.Timestamp, rc_pre: str, a2: float, b2: float):
    """
    Lafourche-style segmented fill:
      - pre-change: original stage-based RC
      - post-change: fitted linear model
    """
    y = Q_orig.copy()
    tags = pd.Series("", index=Q_orig.index, dtype=object)

    for i, date in enumerate(Q_orig.index):
        if pd.isna(Q_orig.iat[i]):
            xv = X.iat[i]
            if pd.notna(xv):
                if date < change_ts:
                    yhat = _safe_eval_rc_stage_only(rc_pre, float(xv))
                else:
                    yhat = float(_line_eval(a2, b2, float(xv)))
                if pd.notna(yhat):
                    y.iat[i] = yhat
                    tags.iat[i] = "Rating Curve"

    return y, tags

# ---------- plotting ----------
def _plot_hq_split(H_pre, Q_pre, H_post, Q_post, rc_pre, a2, b2, station_title, out_path):
    x_all = pd.concat([H_pre, H_post]).dropna()
    if x_all.empty:
        print("Skipping HQ plot (no finite x).")
        return

    x_min = np.nanpercentile(x_all, 1)
    x_max = np.nanpercentile(x_all, 99)
    x_grid = np.linspace(x_min, x_max, 300)

    y_pre = np.array([_safe_eval_rc_stage_only(rc_pre, xv) for xv in x_grid])
    y_post = _line_eval(a2, b2, x_grid)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(H_pre, Q_pre, s=18, c="#1f77b4", alpha=0.75, label="Pre-change points", edgecolors="none")
    ax.scatter(H_post, Q_post, s=18, c="#ff7f0e", alpha=0.75, label="Post-change points", edgecolors="none")
    ax.plot(x_grid, y_pre, color="#1f77b4", lw=2.5, label="Original pre-change RC")
    ax.plot(x_grid, y_post, color="#ff7f0e", lw=2.5, label="Refit post-change line")
    ax.set_xlabel("Stage (H)")
    ax.set_ylabel("Discharge (Q)")
    ax.set_title(station_title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved HQ plot:", out_path)

def _plot_timeseries(df_out: pd.DataFrame, png_out: Path, station_title: str):
    fig, ax = plt.subplots(figsize=(16, 6))
    plot_cat = df_out["Discharge_filled_method"].astype(str)

    method_colors = {
        "Original": "#000000",
        "Interpolated": "#00bcd4",
        "Rating Curve": "#f39c12",
        "Long Gap Interp": "#6a1b9a",
        "Unknown": "#7f7f7f",
    }

    for label, color in method_colors.items():
        sel = plot_cat.eq(label)
        if sel.any():
            ax.scatter(
                df_out.loc[sel, "Date"],
                df_out.loc[sel, "Discharge (m3/s)"],
                s=18 if label == "Original" else 24,
                color=color,
                label=label,
                linewidths=0,
                alpha=0.95,
            )

    y = pd.to_numeric(df_out["Discharge (m3/s)"], errors="coerce").to_numpy()
    m = np.isfinite(y)
    if m.any():
        y_min = float(np.nanmin(y[m]))
        y_max = float(np.nanmax(y[m]))
        span = max(y_max - y_min, 1.0)
        pad = 0.05 * span
        ax.set_ylim(y_min - pad, y_max + pad)

    ax.set_title(station_title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Discharge (m3/s)")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(mdates.YearLocator(base=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.legend(loc="upper right", frameon=True)

    fig.savefig(png_out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved plot:", png_out)

# ---------- output helper ----------
def save_filled_output(df_out: pd.DataFrame, out_dir: Path, station_name: str, sid: str, suffix="_rev"):
    out_dir.mkdir(parents=True, exist_ok=True)
    clean = _clean_name(station_name)
    csv_out = out_dir / f"{clean}_{sid}_filled{suffix}.csv"
    png_out = out_dir / f"{clean}_{sid}_Timeseries_Revised{suffix}.png"
    df_out.to_csv(csv_out, index=False, encoding="utf-8")
    _plot_timeseries(df_out, png_out, f"{station_name} ({sid})")
    print("Saved CSV:", csv_out)
    print("Saved plot:", png_out)
    return csv_out, png_out

In [39]:
# ------------------------------------------------------------
# Cell 4 — Bayou Lafourche (uses shared helpers from Cell 3)
# ------------------------------------------------------------

SID = "07381000"
STATION_NAME = "Bayou Lafourche at Thibodaux, LA"
CHANGE_DATE = "2016-01-01"

OUT_DIR = ROOT / "Revised_Rating_Curves"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUT = OUT_DIR / f"Bayou_Lafourche_at_Thibodaux_LA_{SID}_filled_rev.csv"
PNG_OUT = OUT_DIR / f"Bayou_Lafourche_at_Thibodaux_LA_{SID}_Timeseries_Revised_rev.png"
HQ_OUT  = OUT_DIR / f"Bayou_Lafourche_at_Thibodaux_LA_{SID}_HQ_Fits_rev.png"

# Metadata + pre-change RC
meta = station_metadata.get(SID, {}) or {}
station_name = station_names.get(SID, STATION_NAME)
rc_pre = (meta.get("Rating Curve (Python Format)") or "").strip()
if not rc_pre:
    raise ValueError(f"{SID}: missing pre-change RC in metadata.")

# Load originals-only target + donor predictor stage
Q_orig, H_daily_from_csv, idx = _load_base_series_only_original(objs, SID)
X, Qd_dummy, donor_sid, donor_df = _get_donor_series(objs, SID, idx, H_daily_from_csv)

# Pre/post masks from originals only
change_ts = pd.to_datetime(CHANGE_DATE)
pre_mask  = (idx <  change_ts) & Q_orig.notna() & X.notna()
post_mask = (idx >= change_ts) & Q_orig.notna() & X.notna()

if pre_mask.sum() < 2:
    raise ValueError(f"{SID}: insufficient pre-change originals to plot/validate.")
if post_mask.sum() < 2:
    raise ValueError(f"{SID}: insufficient post-change originals to fit line.")

# Fit post-change line
a2, b2 = _fit_linear(X[post_mask].to_numpy(), Q_orig[post_mask].to_numpy())
print(f"{SID}: donor={donor_sid}  post-fit={a2:.6g}*H + {b2:.6g}  Npost={int(post_mask.sum())}")

# HQ review plot
_plot_hq_split(
    X[pre_mask], Q_orig[pre_mask],
    X[post_mask], Q_orig[post_mask],
    rc_pre, a2, b2,
    f"{station_name} ({SID})",
    HQ_OUT,
)

# Fill all missing with segmented model
y_rc, tags = fill_all_missing_with_split_models(Q_orig, X, change_ts, rc_pre, a2, b2)

# Then interpolation passes
y_filled, tags = fill_short_then_long_interp(y_rc, tags, short_gap_days=SHORT_GAP_DAYS)
tags.loc[Q_orig.notna()] = "Original"

# Save
df_out = pd.DataFrame({
    "Date": idx,
    "Discharge (m3/s)": y_filled.values,
    "Discharge_filled_method": tags.values,
})
df_out.to_csv(CSV_OUT, index=False, encoding="utf-8")
print("Saved CSV:", CSV_OUT)

_plot_timeseries(df_out, PNG_OUT, f"{station_name} ({SID})")
print("Saved plot:", PNG_OUT)
print("Method counts:\n", df_out["Discharge_filled_method"].value_counts(dropna=False))

07381000: donor=07381000  post-fit=6.40481*H + 4.34788  Npost=3421
Saved HQ plot: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lafourche_at_Thibodaux_LA_07381000_HQ_Fits_rev.png
Saved CSV: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lafourche_at_Thibodaux_LA_07381000_filled_rev.csv
Saved plot: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lafourche_at_Thibodaux_LA_07381000_Timeseries_Revised_rev.png
Saved plot: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lafourche_at_Thibodaux_LA_07

In [40]:
# ------------------------------------------------------------
# Cell 5 — Bayou Lacassine (HQ-only robust path, shared helpers)
# ------------------------------------------------------------

SID = "08012470"
STATION_NAME = "Bayou Lacassine near Lake Arthur, LA"
FALLBACK_STATION_NAME = STATION_NAME
PREFER_DV = True

OUT_DIR = ROOT / "Revised_Rating_Curves"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FULL_START = "2006-01-01"
FULL_END   = "2025-09-01"

# Build daily originals-only target + donor H/Q using robust daily path
Q_orig, H, Qd, donor_sid = _build_daily_original_target_and_donor_hq(
    objs,
    SID=SID,
    start=FULL_START,
    end=FULL_END,
    prefer_dv=PREFER_DV,
    short_gap_days=SHORT_GAP_DAYS,
)

# Fallback if DV preference leaves too few overlap points
if pd.concat([Q_orig, H, Qd], axis=1).dropna().shape[0] < 12 and PREFER_DV:
    Q_orig, H, Qd, donor_sid = _build_daily_original_target_and_donor_hq(
        objs,
        SID=SID,
        start=FULL_START,
        end=FULL_END,
        prefer_dv=False,
        short_gap_days=SHORT_GAP_DAYS,
    )

# Fit HQ-only RC
rc_all, nfit, nse = fit_hq_only_rc(Q_orig, H, Qd)
print(f"{SID}: donor={donor_sid}")
print(f"{SID} HQ-only fit: N={nfit}, NSE={nse:.3f}")
print("RC_all:", rc_all)

# RC on long gaps only, then interpolation
y_rc, tags = fill_long_gaps_with_rc(
    Q_orig, H, Qd, rc_all,
    short_gap_days=SHORT_GAP_DAYS,
    tag_name="Rating Curve",
)

y_filled, tags = fill_short_then_long_interp(
    y_rc, tags, short_gap_days=SHORT_GAP_DAYS
)

# Preserve originals label
tags.loc[Q_orig.notna()] = "Original"

# Save
station_name = station_names.get(SID, FALLBACK_STATION_NAME)
df_out = pd.DataFrame({
    "Date": Q_orig.index,
    "Discharge (m3/s)": y_filled.values,
    "Discharge_filled_method": tags.values,
})

csv_out = OUT_DIR / f"{_clean_name(station_name)}_{SID}_filled_rev.csv"
png_out = OUT_DIR / f"{_clean_name(station_name)}_{SID}_Timeseries_Revised_rev.png"

df_out.to_csv(csv_out, index=False, encoding="utf-8")
print("Saved CSV:", csv_out)

_plot_timeseries(df_out, png_out, f"{station_name} ({SID})")
print("Saved plot:", png_out)
print("Method counts:\n", df_out["Discharge_filled_method"].value_counts(dropna=False))

08012470: donor=08012150
08012470 HQ-only fit: N=753, NSE=0.717
RC_all: 16.4485*H + 0.238581*Q + -0.000287596*Q**2 + -9.13151
Saved CSV: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lacassine_near_Lake_Arthur_LA_08012470_filled_rev.csv
Saved plot: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lacassine_near_Lake_Arthur_LA_08012470_Timeseries_Revised_rev.png
Saved plot: C:\Users\p00278065\Coastal Master Plan 2029\ICM_pre-processing\CMP_ICM_pre-processing_CLONE_20260805\ICM_pre-processing\observed_tributary_flows\Revised_Rating_Curves\Bayou_Lacassine_near_Lake_Arthur_LA_08012470_Timeseries_Revised_rev.png
Method counts:
 Discharge_filled_method
Rating Curve       6241
Original            753
Long Gap Interp     150
Interpolated         4